# Lab 15 · Connect agent frameworks to models and tools

Run the same docs agent in Microsoft Agent Framework, the OpenAI Agents SDK, and LangChain with LangGraph, with every
model call and every MCP tool call going through your AI Gateway. This notebook is the code track of the
[lab guide](README.md): it runs the guide's three samples in [infra/](infra/) unchanged, with the gateway settings
in their environment instead of a `.env` file, then proves that the gateway is in the path.

**Get a key → Point the samples at the gateway → Run three frameworks → Prove the gateway is in the path**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Model endpoint `<gateway-url>/default/models/openai/v1` | The OpenAI-compatible endpoint. Each framework's model client sends the runtime key as its API key, and calls the Responses API. |
| MCP endpoint `<gateway-url>/default/toolservers/learn/mcp` | The `learn` MCP server from lab 04. Each framework's MCP client sends the same key in the `api-key` header. |
| Alias `gpt-5-mini` | The model alias from lab 02. The agents name the alias, never the provider's deployment. |
| Runtime API key `docs-agent` | The agent's own key, so its traffic is easy to find in Monitoring. |
| [infra/agent_framework_agent.py](infra/agent_framework_agent.py), [infra/openai_agents_agent.py](infra/openai_agents_agent.py), [infra/langgraph_agent.py](infra/langgraph_agent.py) | The three samples. Each one loads the MCP tools, asks one question, and prints the answer. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, keys, and retries. |

![Agents call models and MCP tools through AI Gateway](../assets/diagrams/architecture.svg)

## Prerequisites

- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`, with
  Contributor on the gateway's resource group, so the notebook can create a key.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions. The samples were
  tested together on Python 3.13 with the versions in [infra/requirements.txt](infra/requirements.txt).
- [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) has run, so the `gpt-5-mini` alias
  exists, and [04-federate-mcp-tools.ipynb](../04-federate-mcp-tools/04-federate-mcp-tools.ipynb) has run with
  `keep_for_lab_05 = True`, so the `learn` MCP server exists.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it reuses the same key.

In [ ]:
%pip install -q -r infra/requirements.txt azure-identity requests

### 0. Initialize the lab

The defaults match notebooks 01, 02, and 04, so `utils.Lab` finds the gateway without edits. If `pip` upgraded a
package that the kernel had already loaded, restart the kernel before you run this cell.

In [ ]:
import os
import subprocess
import sys
import time

import requests
from IPython.display import display

sys.path.insert(0, "../shared")   # helpers and brand styles that every lab notebook shares
import brand
import utils

lab = utils.Lab("15-agent-frameworks")
alias       = "gpt-5-mini"   # model alias from lab 02
server_name = "learn"        # MCP server from lab 04
key_name    = "docs-agent"   # the agent's own runtime key

### 1. Check the alias and the MCP server, and get a key

The agents need a callable alias and an MCP server. The cell checks that both exist, then creates the agent's own
runtime key, `docs-agent`, or reuses it, and waits until the gateway accepts it: a new key can take a minute to work.
Monitoring and cost limits can then tell this agent apart from other callers.

In [ ]:
alias_resource  = lab.arm("GET", f"{lab.workspace_id}/aliases/{alias}")
server_resource = lab.arm("GET", f"{lab.workspace_id}/toolServers/{server_name}")
assert alias_resource, f"Alias {alias} not found. Run notebook 02 first."
assert server_resource, f"MCP server {server_name} not found. Run notebook 04 with keep_for_lab_05 = True first."

api_key, created_key = lab.api_key(key_name)
mcp_url = f"{lab.gateway_url}/default/toolservers/{server_name}/mcp"


def mcp_status():
    """Send an MCP initialize with the key: 200 once the gateway accepts it."""
    return requests.post(mcp_url, timeout=60, headers={
        "api-key": api_key, "Content-Type": "application/json", "Accept": "application/json, text/event-stream"},
        json={"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {
            "protocolVersion": "2025-03-26", "capabilities": {}, "clientInfo": {"name": "aigateway-labs", "version": "1.0"}}},
    ).status_code


assert utils.wait_for_status(mcp_status, 200) == 200, "The key doesn't work yet. Rerun the cell in a minute."
display(brand.info("Gateway resources the agents use", [
    ("Alias", brand.mono(alias)),
    ("MCP server", brand.mono(server_name)),
    ("Model endpoint", brand.mono(f"{lab.gateway_url}/default/models/openai/v1")),
    ("MCP endpoint", brand.mono(mcp_url)),
]))
display(brand.key(key_name, api_key, created_key))

### 2. Point the samples at the gateway

The samples read four settings, from `.env` or from the environment, and a value in the environment wins. The cell
passes the settings to each sample's process, so the key never lands in a file. `run_sample` runs one sample from
the `infra` folder with the kernel's Python, and returns its answer.

In [ ]:
settings = {
    "AI_GATEWAY_URL":        lab.gateway_url,
    "AI_GATEWAY_MODEL":      alias,
    "AI_GATEWAY_API_KEY":    api_key,
    "AI_GATEWAY_MCP_SERVER": server_name,
}


def run_sample(script, key=None):
    """Run one sample, and return (exit code, header line, answer, stderr, seconds)."""
    env = {**os.environ, **settings, **({"AI_GATEWAY_API_KEY": key} if key else {})}
    start = time.time()
    status = brand.live(brand.note(f"Running {script}", "wait"))
    done = subprocess.run([sys.executable, script], cwd="infra", env=env, capture_output=True, text=True, timeout=600)
    seconds = round(time.time() - start)
    status.update(brand.note(f"Ran {script} in {seconds} s", "neutral"))
    header, _, answer = done.stdout.strip().partition("\n")
    return done.returncode, header, answer.strip(), done.stderr, seconds


def show_answer(framework, script):
    exit_code, header, answer, stderr, seconds = run_sample(script)
    assert exit_code == 0, stderr[-2000:]
    cites = "learn.microsoft.com" in answer
    display(brand.reply(answer, f"{framework} agent", model=alias, sub=header,
                        badge=brand.chip("Cites Microsoft Learn", "ok") if cites else brand.chip("No citation", "warn")))
    assert answer, "The agent returned an empty answer"
    return cites


display(brand.info("Settings each sample gets", [
    (name, brand.masked(value) if name == "AI_GATEWAY_API_KEY" else brand.mono(value)) for name, value in settings.items()
]))

### 3. Run the Microsoft Agent Framework agent

`OpenAIChatClient` points at the gateway's model endpoint, and `MCPStreamableHTTPTool` loads the `learn` tools from
the gateway's MCP endpoint. The agent searches Microsoft Learn, then answers with the URLs it used. The answer
differs between runs.

In [ ]:
cites = {"Microsoft Agent Framework": show_answer("Microsoft Agent Framework", "agent_framework_agent.py")}

### 4. Run the OpenAI Agents SDK agent

`OpenAIResponsesModel` wraps an `AsyncOpenAI` client that points at the gateway, and `MCPServerStreamableHttp`
connects to the same MCP endpoint.

In [ ]:
cites["OpenAI Agents SDK"] = show_answer("OpenAI Agents SDK", "openai_agents_agent.py")

### 5. Run the LangGraph agent

`ChatOpenAI` with `use_responses_api=True` points at the gateway, and `MultiServerMCPClient` from
langchain-mcp-adapters loads the tools. Its header line also says how many tools it loaded.

In [ ]:
cites["LangGraph"] = show_answer("LangGraph", "langgraph_agent.py")

display(brand.table("Three frameworks, one gateway", ["Framework", "Model", "MCP server", "Cites Microsoft Learn"],
                    [(name, brand.mono(alias), brand.mono(server_name),
                      brand.chip("Yes", "ok") if ok else brand.chip("No", "warn")) for name, ok in cites.items()]))
assert any(cites.values()), "No agent cited Microsoft Learn: check that the learn server publishes tools"

### 6. Prove the gateway is in the path

Run the LangGraph sample again with a wrong key. The gateway rejects the agent's first call with `401`, before any
model or tool runs: each sample connects to the MCP server before it calls the model.

If you set up monitoring in [lab 13](../13-monitoring/README.md), wait a few minutes and open **Monitoring** in the
portal. **Model calls** and **Tool calls** both grew, and **Tool usage by caller identity** shows the `docs-agent` key.

In [ ]:
exit_code, _, _, stderr, _ = run_sample("langgraph_agent.py", key="not-a-key")
rejected = 401 if "401" in stderr else exit_code
detail = next((line.strip() for line in reversed(stderr.splitlines()) if "401 Unauthorized" in line), stderr.strip()[-200:])

display(brand.checks("Prove the gateway is in the path", [
    ("LangGraph agent with a wrong key", rejected, 401, detail),
]))
assert exit_code != 0 and rejected == 401, stderr[-2000:]

### 7. Clean up resources

The cell revokes the `docs-agent` key if this notebook created it. It keeps the gateway, the alias, and the `learn`
MCP server, because other labs reuse them. The samples wrote no files.

In [ ]:
if created_key:
    lab.delete(f"{lab.gateway_id}/apiKeys/{key_name}")
else:
    display(brand.note(f"Kept key {key_name}, which existed before this notebook ran", "neutral"))

## Troubleshooting

- **`ModuleNotFoundError` in a sample**: the packages aren't in the kernel's environment. Rerun the `%pip` cell, then
  restart the kernel.
- **`401` from the model or MCP endpoint with the right key**: a new key can take a minute to work. Rerun the cell.
- **`404` from the MCP endpoint**: `server_name` doesn't match an MCP server. Rerun notebook 04 with
  `keep_for_lab_05 = True`, or set `server_name` to one of your servers.
- **A `4xx` error that mentions the model**: `alias` isn't a callable alias. Copy it from the model's **Aliases** tab.
- **The agent answers without citing any URL**: the MCP server publishes no tools, or tool controls block them. Check
  the server's tool controls from lab 05.
- **`429` responses**: a token limit or rate limit applies to the alias, the MCP server, or the gateway. Wait for
  `Retry-After`, or raise the limit.

For the other errors in this lab, see [Troubleshooting](../TROUBLESHOOTING.md#lab-15-agent-frameworks).

## Next steps

- Run an Agent Framework agent in Microsoft Foundry with the same gateway wiring in
  [lab 16](../16-foundry-hosted-agent/README.md) and its notebook [16-foundry-hosted-agent.ipynb](../16-foundry-hosted-agent/16-foundry-hosted-agent.ipynb).
- Replace the agent's key with an identity in [lab 11](../11-entra-rbac/README.md) and its notebook [11-entra-rbac.ipynb](../11-entra-rbac/11-entra-rbac.ipynb).
- Docs: [Connect agents to AI Gateway](https://ai.gateway.azure.com/docs/agent-access).
- Read this lab on the labs site: [https://azure.github.io/aigateway/labs/15-agent-frameworks/](https://azure.github.io/aigateway/labs/15-agent-frameworks/).